#Mounting Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
cd /content/drive/MyDrive/33project/04.메디컬/02.VoxelMorph

/content/drive/MyDrive/33project/04.메디컬/02.VoxelMorph


#Installing Dependencies

In [ ]:
!pip3 install pystrum
!pip3 install --upgrade --no-cache-dir gdown

  Preparing metadata (setup.py) ... done
  Created wheel for pystrum: filename=pystrum-0.4-py3-none-any.whl size=19569 sha256=d00bd75d7c396b085e30e7c7dd84d6d04cefbb78ec1a1566a649d3ae65078e31
  Stored in directory: /root/.cache/pip/wheels/e2/95/8d/18fd7cda9f20e4c15ef05a975cccaab741e2f63b0256ff1ce1
Successfully built pystrum
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 5.3 MB/s eta 0:00:00
  Attempting uninstall: gdown
    Found existing installation: gdown 5.2.2
    Uninstalling gdown-5.2.2:
      Successfully uninstalled gdown-5.2.2


#Importing Libraries

In [ ]:
from data.datasets import*
from torch.utils.tensorboard import SummaryWriter
import os, utils, glob, losses, sys
from torch.utils.data import DataLoader
from data import trans
import numpy as np
import torch
from torchvision import transforms
from torch import optim
import matplotlib.pyplot as plt
from natsort import natsorted
from models import STNFunction_ND_BCXYZ,not_normalized_identity_map,identity_map,identity_map_for_reproduce,gen_identity_map,scale_map,scale_map_grad,STN_ND_BCXYZ,convBlock,SpatialTransformer
from data import datasets, trans
import torch.nn as nn

import torch.nn.functional as F

from functools import partial
import finite_differences as fdt

#Defining Variables

In [ ]:
model='experiments/Vxm'
dataset='IXI_data'
dim = 3
reproduce_paper_result = False

#Downloading PreTrained Model

In [ ]:
if os.path.exists(model) is False:
  print("Downloading Model")
  os.makedirs(model)
  %cd experiments/Vxm
  !gdown https://drive.google.com/u/0/uc?id=1Dv6Z1MK_JU6dveGHu6jkY3VRUuiXRFG8&export=download
  %cd ../../
else:
  print('Model Already Downloaded')

/content/drive/MyDrive/33project/04.메디컬/02.VoxelMorph/experiments/Vxm
Downloading...
From: https://drive.google.com/uc?id=1Dv6Z1MK_JU6dveGHu6jkY3VRUuiXRFG8
To: /content/drive/MyDrive/33project/04.메디컬/02.VoxelMorph/experiments/Vxm/VoxelMorph_diff_Validation_dsc0.591.pth.tar
100% 3.72M/3.72M [00:00<00:00, 29.6MB/s]
/content/drive/MyDrive/33project/04.메디컬/02.VoxelMorph


#Downloading Dataset

In [ ]:
if os.path.exists(dataset) is False:
  print('Downloading Dataset')
  !gdown https://drive.google.com/u/0/uc?id=1-VQewCVNj5eTtc3eQGhTM2yXBQmgm8Ol&export=download
  !unzip IXI_data.zip
else:
  print('Dataset Already Downloaded')

Using cookies from /root/.cache/gdown/cookies.txt
Downloading...
From (original): https://drive.google.com/uc?id=1-VQewCVNj5eTtc3eQGhTM2yXBQmgm8Ol
From (redirected): https://drive.google.com/uc?id=1-VQewCVNj5eTtc3eQGhTM2yXBQmgm8Ol&confirm=t&uuid=1482867e-fb25-4d22-887a-a2c7eaa52be9
To: /content/drive/MyDrive/33project/04.메디컬/02.VoxelMorph/IXI_data.zip
100% 1.54G/1.54G [00:27<00:00, 55.1MB/s]
Archive:  IXI_data.zip
   creating: IXI_data/
  inflating: IXI_data/atlas.pkl      
  inflating: IXI_data/dataset_info.txt  
  inflating: IXI_data/license.txt    
   creating: IXI_data/Test/
  inflating: IXI_data/Test/subject_1.pkl  
  inflating: IXI_data/Test/subject_110.pkl  
  inflating: IXI_data/Test/subject_112.pkl  
  inflating: IXI_data/Test/subject_115.pkl  
  inflating: IXI_data/Test/subject_13.pkl  
  inflating: IXI_data/Test/subject_135.pkl  
  inflating: IXI_data/Test/subject_139.pkl  
  inflating: IXI_data/Test/subject_155.pkl  
  inflating: IXI_data/Test/subject_156.pkl  
  inflat


#Defining Model

In [ ]:
class Bilinear(nn.Module):
    """
   Spatial transform function for 1D, 2D, and 3D. In BCXYZ format (this IS the format used in the current toolbox).
   """

    def __init__(self, zero_boundary=False, using_scale=False, mode='bilinear'):
        """
        Constructor
        :param ndim: (int) spatial transformation of the transform
        """
        super(Bilinear, self).__init__()
        self.zero_boundary = 'zeros' if zero_boundary else 'border'
        self.using_scale = using_scale
        """ scale [-1,1] image intensity into [0,1], this is due to the zero boundary condition we may use here """
        self.mode = mode

    def forward_stn(self, input1, input2):
        input2_ordered = torch.zeros_like(input2)
        input2_ordered[:, 0, ...] = input2[:, 2, ...]
        input2_ordered[:, 1, ...] = input2[:, 1, ...]
        input2_ordered[:, 2, ...] = input2[:, 0, ...]

        output = torch.nn.functional.grid_sample(input1, input2_ordered.permute([0, 2, 3, 4, 1]), mode=self.mode,
                                                     padding_mode=self.zero_boundary, align_corners=True)
        return output

    def forward(self, input1, input2):
        """
        Perform the actual spatial transform
        :param input1: image in BCXYZ format
        :param input2: spatial transform in BdimXYZ format
        :return: spatially transformed image in BCXYZ format
        """
        if self.using_scale:

            output = self.forward_stn((input1 + 1) / 2, input2)
            # print(STNVal(output, ini=-1).sum())
            return output * 2 - 1
        else:
            output = self.forward_stn(input1, input2)
            # print(STNVal(output, ini=-1).sum())
            return output


class VoxelMorphMICCAI2019(nn.Module):
    """
    unet architecture for voxelmorph models presented in the MICCAI 2019 paper.
    You may need to modify this code (e.g., number of layers) to suit your project needs.
    :param vol_size: volume size. e.g. (256, 256, 256)
    :param enc_nf: list of encoder filters. right now it needs to be 1x4.
           e.g. [16,32,32,32]
    :param dec_nf: list of decoder filters. right now it must be 1x6 (like voxelmorph-1) or 1x7 (voxelmorph-2)
    :return: the reg_model
    """
    def __init__(self, img_sz, image_sigma=0.01, prior_lambda=10, prior_lambda_mean=10):
        super(VoxelMorphMICCAI2019, self).__init__()

        enc_filters = [16, 32, 32, 32, 32]
        #dec_filters = [32, 32, 32, 8, 8]
        dec_filters = [32, 32, 32, 32, 16]
        self.enc_filter = enc_filters
        self.dec_filter = dec_filters
        input_channel =2
        output_channel= 3
        self.input_channel = input_channel
        self.output_channel = output_channel
        self.img_sz = img_sz
        self.low_res_img_sz = [int(x/2) for x in img_sz]
        self.spacing = 1. / ( np.array(img_sz) - 1)
        self.int_steps = 7

        self.image_sigma = image_sigma#opt_voxelmorph[('image_sigma',0.02,'image_sigma')]
        self.prior_lambda = prior_lambda#opt_voxelmorph[('lambda_factor_in_vmr',50,'lambda_factor_in_vmr')]
        self.prior_lambda_mean = prior_lambda_mean#opt_voxelmorph[('lambda_mean_factor_in_vmr',50,'lambda_mean_factor_in_vmr')]
        self.flow_vol_shape = self.low_res_img_sz
        self.D = self._degree_matrix(self.flow_vol_shape)
        self.D = (self.D).cuda()# 1, 96, 40,40 3'
        self.loss_fn =  None

        self.id_transform = gen_identity_map(self.img_sz, 1.0).cuda()
        self.id_transform  =self.id_transform.view([1]+list(self.id_transform.shape))

        """to compatiable to the mesh setting in voxel morph"""
        self.low_res_id_transform = gen_identity_map(self.img_sz, 0.5, normalized=False).cuda()
        self.encoders = nn.ModuleList()
        self.decoders = nn.ModuleList()
        #self.bilinear = Bilinear(zero_boundary=True)
        self.bilinear = STN_ND_BCXYZ(np.array([1.,1.,1.]),zero_boundary=True)
        self.bilinear_img = Bilinear(zero_boundary=True)
        for i in range(len(enc_filters)):
            if i==0:
                self.encoders.append(convBlock(input_channel, enc_filters[i], stride=1, bias=True))
            else:
                self.encoders.append(convBlock(enc_filters[i-1], enc_filters[i], stride=2, bias=True))

        self.decoders.append(convBlock(enc_filters[-1], dec_filters[0], stride=1, bias=True))
        self.decoders.append(convBlock(dec_filters[0] + enc_filters[3],dec_filters[1], stride=1, bias=True))
        self.decoders.append(convBlock(dec_filters[1] + enc_filters[2],dec_filters[2], stride=1, bias=True))
        self.decoders.append(convBlock(dec_filters[2] + enc_filters[1],dec_filters[3], stride=1, bias=True))
        self.decoders.append(convBlock(dec_filters[3], dec_filters[4],stride=1, bias=True))

        self.flow_mean =  nn.Conv3d(dec_filters[-1], output_channel, kernel_size=3, stride=1, padding=1, bias=True)
        self.flow_sigma =  nn.Conv3d(dec_filters[-1], output_channel, kernel_size=3, stride=1, padding=1, bias=True)
        self.flow_mean.weight.data.normal_(0.,1e-5)
        self.flow_sigma.weight.data.normal_(0.,1e-10)
        self.flow_sigma.bias.data = torch.Tensor([-10]*3)
        self.print_count=0
        # identity transform for computing displacement

    def scale_map(self,map, spacing):
        """
        Scales the map to the [-1,1]^d format
        :param map: map in BxCxXxYxZ format
        :param spacing: spacing in XxYxZ format
        :return: returns the scaled map
        """
        sz = map.size()
        map_scaled = torch.zeros_like(map)
        ndim = len(spacing)

        # This is to compensate to get back to the [-1,1] mapping of the following form
        # id[d]*=2./(sz[d]-1)
        # id[d]-=1.

        for d in range(ndim):
            if sz[d + 2] > 1:
                map_scaled[:, d, ...] = map[:, d, ...] * (2. / (sz[d + 2] - 1.) / spacing[d])
            else:
                map_scaled[:, d, ...] = map[:, d, ...]

        return map_scaled

    def set_loss_fn(self, loss_fn):
        """ set loss function"""
        pass

    def forward(self, x):
        source, target = x
        self.__do_some_clean()
        affine_map = self.id_transform.clone()
        x_enc_1 = self.encoders[0](torch.cat((source, target), dim=1))
        # del input
        x_enc_2 = self.encoders[1](x_enc_1)
        x_enc_3 = self.encoders[2](x_enc_2)
        x_enc_4 = self.encoders[3](x_enc_3)
        x_enc_5 = self.encoders[4](x_enc_4)

        x = self.decoders[0](x_enc_5)
        x = F.interpolate(x,scale_factor=2,mode='trilinear')
        x = torch.cat((x, x_enc_4),dim=1)
        x = self.decoders[1](x)
        x = F.interpolate(x, scale_factor=2, mode='trilinear')
        x = torch.cat((x, x_enc_3), dim=1)
        x = self.decoders[2](x)
        x = F.interpolate(x, scale_factor=2, mode='trilinear')
        x = torch.cat((x, x_enc_2), dim=1)
        x = self.decoders[3](x)
        x = self.decoders[4](x)
        flow_mean = self.flow_mean(x)
        log_sigma = self.flow_sigma(x)
        noise = torch.randn(flow_mean.shape).cuda()
        if self.training:
            flow = flow_mean + torch.exp(log_sigma / 2.0) * noise
        else:
            flow = flow_mean #+ 1 * noise

        for _ in range(self.int_steps):
            deform_field = flow + self.low_res_id_transform
            flow_1 = self.bilinear(flow, deform_field)
            flow = flow_1 + flow
        disp_field = F.interpolate(flow, scale_factor=2, mode='trilinear')
        disp_field = self.scale_map(disp_field, np.array([1,1,1]))
        deform_field = disp_field + affine_map
        warped_source = self.bilinear_img(source, deform_field)
        self.res_flow_mean  = flow
        self.res_log_sigma = log_sigma
        self.warped = warped_source
        self.target = target
        self.source = source

        return warped_source, deform_field, disp_field

    def check_if_update_lr(self):
        return False, None

    def get_extra_to_plot(self):
        return None, None
    def __do_some_clean(self):
        self.res_flow_mean = None
        self.res_log_sigma = None
        self.warped = None
        self.target = None
        self.source = None

    def scale_reg_loss(self,):
        reg = self.kl_loss()
        return reg

    def get_sim_loss(self,):
        loss = self.recon_loss()
        return loss

    def _adj_filt(self, ndims):
        """
        compute an adjacency filter that, for each feature independently,
        has a '1' in the immediate neighbor, and 0 elsewehre.
        so for each filter, the filter has 2^ndims 1s.
        the filter is then setup such that feature i outputs only to feature i
        """

        # inner filter, that is 3x3x...
        filt_inner = np.zeros([3] * ndims)  # 3 3 3
        for j in range(ndims):
            o = [[1]] * ndims
            o[j] = [0, 2]
            filt_inner[np.ix_(*o)] = 1

        # full filter, that makes sure the inner filter is applied
        # ith feature to ith feature
        filt = np.zeros([ndims, ndims] + [3] * ndims)  # 3 3 3 3  ##!!!!!!!! in out w h d
        for i in range(ndims):
            filt[i, i, ...] = filt_inner  ##!!!!!!!!

        return filt

    def _degree_matrix(self, vol_shape):
        # get shape stats
        ndims = len(vol_shape)
        sz = [ndims,*vol_shape]  # 96 96 40 3  ##!!!!!!!!

        # prepare conv kernel
        conv_fn = F.conv3d  ##!!!!!!!!

        # prepare tf filter
        z = torch.ones([1] + sz)  # 1 96 96 40 3
        filt_tf = torch.Tensor(self._adj_filt(ndims))  # 3 3 3 3 ##!!!!!!!!
        strides = [1] * (ndims)  ##!!!!!!!!
        return conv_fn(z, filt_tf, padding= 1, stride =strides)  ##!!!!!!!!

    def prec_loss(self, disp):  ##!!!!!!!!
        """
        a more manual implementation of the precision matrix term
                mu * P * mu    where    P = D - A
        where D is the degree matrix and A is the adjacency matrix
                mu * P * mu = 0.5 * sum_i mu_i sum_j (mu_i - mu_j) = 0.5 * sum_i,j (mu_i - mu_j) ^ 2
        where j are neighbors of i
        Note: could probably do with a difference filter,
        but the edges would be complicated unless tensorflow allowed for edge copying
        """
        fd = fdt.FD_torch(np.array([1., 1., 1.]))
        dfx = fd.dXc(disp[:, 0, ...])
        dfy = fd.dYc(disp[:, 1, ...])
        dfz = fd.dZc(disp[:, 2, ...])
        l2 = dfx ** 2 + dfy ** 2 + dfz ** 2
        reg = l2.mean()
        return reg * 0.5

    def kl_loss(self):
        """
        KL loss
        y_pred is assumed to be D*2 channels: first D for mean, next D for logsigma
        D (number of dimensions) should be 1, 2 or 3
        y_true is only used to get the shape
        """
        # prepare inputs
        ndims = 3
        flow_mean = self.res_flow_mean
        log_sigma = self.res_log_sigma

        # compute the degree matrix (only needs to be done once)
        # we usually can't compute this until we know the ndims,
        # which is a function of the data

        # sigma terms
        sigma_term = self.prior_lambda * self.D * torch.exp(log_sigma) - log_sigma  ##!!!!!!!!
        sigma_term = torch.mean(sigma_term)  ##!!!!!!!!

        # precision terms
        # note needs 0.5 twice, one here (inside self.prec_loss), one below
        prec_term = self.prior_lambda_mean * self.prec_loss(flow_mean)  # this is the jacobi loss
        #if self.print_count%10==0:
        #    print("the loss of neg log_sigma is {},  the sigma term is {}, the loss of the prec term is {}".format((-log_sigma).mean().item(),sigma_term,prec_term))

        # combine terms
        return 0.5 * ndims * (sigma_term + prec_term)  # ndims because we averaged over dimensions as well

    def recon_loss(self):
        """ reconstruction loss """
        y_pred = self.warped
        y_true = self.target
        return 1. / (self.image_sigma ** 2) * torch.mean((y_true - y_pred)**2)  ##!!!!!!!!

    def get_loss(self):
        sim_loss = self.get_sim_loss()
        reg_loss = self.scale_reg_loss()
        return sim_loss+ reg_loss

    def get_inverse_map(self,):
        print("VoxelMorph approach doesn't support analytical computation of inverse map")
        print("Instead, we compute it's numerical approximation")
        _, inverse_map = self.forward(self.target, self.source)
        return inverse_map

    def weights_init(self):
        for m in self.modules():
            classname = m.__class__.__name__
            if classname.find('Conv') != -1:
                if not m.weight is None:
                    nn.init.xavier_normal_(m.weight.data)
                if not m.bias is None:
                    m.bias.data.zero_()

#Training Script

In [ ]:
import os
import sys
import glob
import numpy as np
import torch
import torch.optim as optim
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms
from natsort import natsorted


class Logger(object):
    def __init__(self, save_dir):
        self.terminal = sys.stdout
        self.log = open(save_dir + "logfile.log", "a")

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)

    def flush(self):
        self.terminal.flush()
        self.log.flush()


def main():
    batch_size = 1

    atlas_dir = 'IXI_data/atlas.pkl'
    train_dir = 'IXI_data/Train/'
    val_dir = 'IXI_data/Val/'

    save_dir = 'VxmDiff/'

    if not os.path.exists('experiments/' + save_dir):
        os.makedirs('experiments/' + save_dir)

    if not os.path.exists('logs/' + save_dir):
        os.makedirs('logs/' + save_dir)

    sys.stdout = Logger('logs/' + save_dir)

    lr = 0.0001
    epoch_start = 0
    max_epoch = 500
    img_size = (160, 192, 224)
    cont_training = False

    '''
    Initialize model
    '''
    model = VoxelMorphMICCAI2019(img_size)
    model.cuda()

    '''
    Initialize spatial transformation function
    '''
    reg_model = Bilinear(
        zero_boundary=True,
        mode='nearest'
    ).cuda()

    for param in reg_model.parameters():
        param.requires_grad = False

    reg_model_bilin = Bilinear(
        zero_boundary=True,
        mode='bilinear'
    ).cuda()

    for param in reg_model_bilin.parameters():
        param.requires_grad = False

    '''
    If continue from previous training
    '''
    if cont_training:
        epoch_start = 0

        model_dir = 'experiments/' + save_dir

        updated_lr = round(
            lr * np.power(
                1 - (epoch_start / max_epoch),
                0.9
            ),
            8
        )

        model_files = natsorted(os.listdir(model_dir))

        best_model = torch.load(
            model_dir + model_files[-1]
        )['state_dict']

        print(
            'Model: {} loaded!'.format(
                model_files[-1]
            )
        )

        model.load_state_dict(best_model)

    else:
        updated_lr = lr

    '''
    Initialize training
    '''
    train_composed = transforms.Compose([
        trans.RandomFlip(0),
        trans.NumpyType(
            (np.float32, np.float32)
        ),
    ])

    val_composed = transforms.Compose([
        trans.Seg_norm(),
        trans.NumpyType(
            (np.float32, np.int16)
        ),
    ])

    train_set = datasets.IXIBrainDataset(
        glob.glob(train_dir + '*.pkl'),
        atlas_dir,
        transforms=train_composed
    )

    val_set = datasets.IXIBrainInferDataset(
        glob.glob(val_dir + '*.pkl'),
        atlas_dir,
        transforms=val_composed
    )

    # 최신 Colab/Jupyter에서 worker traceback이 꼬이지 않도록 0으로 설정
    train_loader = DataLoader(
        train_set,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_set,
        batch_size=1,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        drop_last=True
    )

    optimizer = optim.Adam(
        model.parameters(),
        lr=updated_lr,
        weight_decay=0,
        amsgrad=True
    )

    best_dsc = 0

    # 기존 코드에서는 이 부분이 주석 처리되어 있었는데
    # 아래에서 writer를 사용하므로 반드시 생성해야 함
    writer = SummaryWriter(
        log_dir='logs/' + save_dir
    )

    for epoch in range(epoch_start, max_epoch):
        print('Training Starts')

        '''
        Training
        '''
        loss_all = utils.AverageMeter()
        idx = 0

        for data in train_loader:
            loss_sim_iter = 0
            loss_reg_iter = 0

            idx += 1

            model.train()

            adjust_learning_rate(
                optimizer,
                epoch,
                max_epoch,
                lr
            )

            data = [
                t.cuda(non_blocking=True)
                for t in data
            ]

            x = data[0]
            y = data[1]

            output = model((x, y))

            loss_sim = model.get_sim_loss()
            loss_sim_iter += loss_sim

            loss_reg = model.scale_reg_loss()
            loss_reg_iter += loss_reg

            loss = loss_sim + loss_reg

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            loss_all.update(
                loss.item(),
                y.numel()
            )

            print(
                'Iter {} of {} loss {:.4f}, '
                'Img Sim: {:.6f}, Reg: {:.6f}'.format(
                    idx,
                    len(train_loader),
                    loss.item(),
                    loss_sim_iter.item(),
                    loss_reg_iter.item()
                )
            )

            del output

        writer.add_scalar(
            'Loss/train',
            loss_all.avg,
            epoch
        )

        print(
            'Epoch {} loss {:.4f}'.format(
                epoch,
                loss_all.avg
            )
        )

        '''
        Validation
        '''
        eval_dsc = utils.AverageMeter()

        with torch.no_grad():
            for data in val_loader:
                model.eval()

                data = [
                    t.cuda(non_blocking=True)
                    for t in data
                ]

                x = data[0]
                y = data[1]
                x_seg = data[2]
                y_seg = data[3]

                grid_img = mk_grid_img(
                    8,
                    1,
                    img_size
                )

                _, flow, _ = model((x, y))

                def_out = reg_model(
                    x_seg.float(),
                    flow
                )

                def_grid = reg_model_bilin(
                    grid_img.float(),
                    flow
                )

                dsc = utils.dice_val_VOI(
                    def_out.long(),
                    y_seg.long()
                )

                eval_dsc.update(
                    dsc.item(),
                    x.size(0)
                )

                print(eval_dsc.avg)

        best_dsc = max(
            eval_dsc.avg,
            best_dsc
        )

        save_checkpoint(
            {
                'epoch': epoch + 1,
                'state_dict': model.state_dict(),
                'best_dsc': best_dsc,
                'optimizer': optimizer.state_dict(),
            },
            save_dir='experiments/' + save_dir,
            filename='dsc{:.3f}.pth.tar'.format(
                eval_dsc.avg
            )
        )

        writer.add_scalar(
            'DSC/validate',
            eval_dsc.avg,
            epoch
        )

        plt.switch_backend('agg')

        pred_fig = comput_fig(def_out)
        grid_fig = comput_fig(def_grid)
        x_fig = comput_fig(x_seg)
        tar_fig = comput_fig(y_seg)

        writer.add_figure(
            'Grid',
            grid_fig,
            epoch
        )

        plt.close(grid_fig)

        writer.add_figure(
            'input',
            x_fig,
            epoch
        )

        plt.close(x_fig)

        writer.add_figure(
            'ground truth',
            tar_fig,
            epoch
        )

        plt.close(tar_fig)

        writer.add_figure(
            'prediction',
            pred_fig,
            epoch
        )

        plt.close(pred_fig)

        loss_all.reset()

    writer.close()


def comput_fig(img):
    img = (
        img.detach()
        .cpu()
        .numpy()[0, 0, 48:64, :, :]
    )

    fig = plt.figure(
        figsize=(12, 12),
        dpi=180
    )

    for i in range(img.shape[0]):
        plt.subplot(
            4,
            4,
            i + 1
        )

        plt.axis('off')

        plt.imshow(
            img[i, :, :],
            cmap='gray'
        )

    fig.subplots_adjust(
        wspace=0,
        hspace=0
    )

    return fig


def adjust_learning_rate(
    optimizer,
    epoch,
    MAX_EPOCHES,
    INIT_LR,
    power=0.9
):
    for param_group in optimizer.param_groups:
        param_group['lr'] = round(
            INIT_LR
            * np.power(
                1 - (epoch / MAX_EPOCHES),
                power
            ),
            8
        )


def mk_grid_img(
    grid_step,
    line_thickness=1,
    grid_sz=(160, 192, 224)
):
    grid_img = np.zeros(
        grid_sz
    )

    for j in range(
        0,
        grid_img.shape[1],
        grid_step
    ):
        grid_img[
            :,
            j + line_thickness - 1,
            :
        ] = 1

    for i in range(
        0,
        grid_img.shape[2],
        grid_step
    ):
        grid_img[
            :,
            :,
            i + line_thickness - 1
        ] = 1

    grid_img = grid_img[
        None,
        None,
        ...
    ]

    grid_img = torch.from_numpy(
        grid_img
    ).float().cuda()

    return grid_img


def save_checkpoint(
    state,
    save_dir='models/',
    filename='checkpoint.pth.tar',
    max_model_num=8
):
    os.makedirs(
        save_dir,
        exist_ok=True
    )

    torch.save(
        state,
        os.path.join(
            save_dir,
            filename
        )
    )

    model_lists = natsorted(
        glob.glob(
            os.path.join(
                save_dir,
                '*'
            )
        )
    )

    while len(model_lists) > max_model_num:
        os.remove(
            model_lists[0]
        )

        model_lists = natsorted(
            glob.glob(
                os.path.join(
                    save_dir,
                    '*'
                )
            )
        )


'''
GPU configuration
'''
GPU_iden = 0

GPU_num = torch.cuda.device_count()

print(
    'Number of GPU: '
    + str(GPU_num)
)

for GPU_idx in range(GPU_num):
    GPU_name = torch.cuda.get_device_name(
        GPU_idx
    )

    print(
        '     GPU #'
        + str(GPU_idx)
        + ': '
        + GPU_name
    )

if GPU_num == 0:
    raise RuntimeError(
        'CUDA GPU를 찾을 수 없습니다.'
    )

torch.cuda.set_device(
    GPU_iden
)

GPU_avai = torch.cuda.is_available()

print(
    'Currently using: '
    + torch.cuda.get_device_name(
        GPU_iden
    )
)

print(
    'If the GPU is available? '
    + str(GPU_avai)
)


main()

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



#Inference Script

In [ ]:
import os
import sys
import glob
import numpy as np
import torch
import torch.optim as optim
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms
from natsort import natsorted


class Logger(object):
    def __init__(self, save_dir):
        self.terminal = sys.stdout
        self.log = open(save_dir + "logfile.log", "a")

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)

    def flush(self):
        self.terminal.flush()
        self.log.flush()


def main():
    batch_size = 1

    atlas_dir = 'IXI_data/atlas.pkl'
    train_dir = 'IXI_data/Train/'
    val_dir = 'IXI_data/Val/'

    save_dir = 'VxmDiff/'

    if not os.path.exists('experiments/' + save_dir):
        os.makedirs('experiments/' + save_dir)

    if not os.path.exists('logs/' + save_dir):
        os.makedirs('logs/' + save_dir)

    sys.stdout = Logger('logs/' + save_dir)

    lr = 0.0001
    epoch_start = 0
    max_epoch = 500
    img_size = (160, 192, 224)
    cont_training = False

    '''
    Initialize model
    '''
    model = VoxelMorphMICCAI2019(img_size)
    model.cuda()

    '''
    Initialize spatial transformation function
    '''
    reg_model = Bilinear(
        zero_boundary=True,
        mode='nearest'
    ).cuda()

    for param in reg_model.parameters():
        param.requires_grad = False

    reg_model_bilin = Bilinear(
        zero_boundary=True,
        mode='bilinear'
    ).cuda()

    for param in reg_model_bilin.parameters():
        param.requires_grad = False

    '''
    If continue from previous training
    '''
    if cont_training:
        epoch_start = 0

        model_dir = 'experiments/' + save_dir

        updated_lr = round(
            lr * np.power(
                1 - (epoch_start / max_epoch),
                0.9
            ),
            8
        )

        model_files = natsorted(os.listdir(model_dir))

        best_model = torch.load(
            model_dir + model_files[-1]
        )['state_dict']

        print(
            'Model: {} loaded!'.format(
                model_files[-1]
            )
        )

        model.load_state_dict(best_model)

    else:
        updated_lr = lr

    '''
    Initialize training
    '''
    train_composed = transforms.Compose([
        trans.RandomFlip(0),
        trans.NumpyType(
            (np.float32, np.float32)
        ),
    ])

    val_composed = transforms.Compose([
        trans.Seg_norm(),
        trans.NumpyType(
            (np.float32, np.int16)
        ),
    ])

    train_set = datasets.IXIBrainDataset(
        glob.glob(train_dir + '*.pkl'),
        atlas_dir,
        transforms=train_composed
    )

    val_set = datasets.IXIBrainInferDataset(
        glob.glob(val_dir + '*.pkl'),
        atlas_dir,
        transforms=val_composed
    )

    # 최신 Colab/Jupyter에서 worker traceback이 꼬이지 않도록 0으로 설정
    train_loader = DataLoader(
        train_set,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_set,
        batch_size=1,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        drop_last=True
    )

    optimizer = optim.Adam(
        model.parameters(),
        lr=updated_lr,
        weight_decay=0,
        amsgrad=True
    )

    best_dsc = 0

    # 기존 코드에서는 이 부분이 주석 처리되어 있었는데
    # 아래에서 writer를 사용하므로 반드시 생성해야 함
    writer = SummaryWriter(
        log_dir='logs/' + save_dir
    )

    for epoch in range(epoch_start, max_epoch):
        print('Training Starts')

        '''
        Training
        '''
        loss_all = utils.AverageMeter()
        idx = 0

        for data in train_loader:
            loss_sim_iter = 0
            loss_reg_iter = 0

            idx += 1

            model.train()

            adjust_learning_rate(
                optimizer,
                epoch,
                max_epoch,
                lr
            )

            data = [
                t.cuda(non_blocking=True)
                for t in data
            ]

            x = data[0]
            y = data[1]

            output = model((x, y))

            loss_sim = model.get_sim_loss()
            loss_sim_iter += loss_sim

            loss_reg = model.scale_reg_loss()
            loss_reg_iter += loss_reg

            loss = loss_sim + loss_reg

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            loss_all.update(
                loss.item(),
                y.numel()
            )

            print(
                'Iter {} of {} loss {:.4f}, '
                'Img Sim: {:.6f}, Reg: {:.6f}'.format(
                    idx,
                    len(train_loader),
                    loss.item(),
                    loss_sim_iter.item(),
                    loss_reg_iter.item()
                )
            )

            del output

        writer.add_scalar(
            'Loss/train',
            loss_all.avg,
            epoch
        )

        print(
            'Epoch {} loss {:.4f}'.format(
                epoch,
                loss_all.avg
            )
        )

        '''
        Validation
        '''
        eval_dsc = utils.AverageMeter()

        with torch.no_grad():
            for data in val_loader:
                model.eval()

                data = [
                    t.cuda(non_blocking=True)
                    for t in data
                ]

                x = data[0]
                y = data[1]
                x_seg = data[2]
                y_seg = data[3]

                grid_img = mk_grid_img(
                    8,
                    1,
                    img_size
                )

                _, flow, _ = model((x, y))

                def_out = reg_model(
                    x_seg.float(),
                    flow
                )

                def_grid = reg_model_bilin(
                    grid_img.float(),
                    flow
                )

                dsc = utils.dice_val_VOI(
                    def_out.long(),
                    y_seg.long()
                )

                eval_dsc.update(
                    dsc.item(),
                    x.size(0)
                )

                print(eval_dsc.avg)

        best_dsc = max(
            eval_dsc.avg,
            best_dsc
        )

        save_checkpoint(
            {
                'epoch': epoch + 1,
                'state_dict': model.state_dict(),
                'best_dsc': best_dsc,
                'optimizer': optimizer.state_dict(),
            },
            save_dir='experiments/' + save_dir,
            filename='dsc{:.3f}.pth.tar'.format(
                eval_dsc.avg
            )
        )

        writer.add_scalar(
            'DSC/validate',
            eval_dsc.avg,
            epoch
        )

        plt.switch_backend('agg')

        pred_fig = comput_fig(def_out)
        grid_fig = comput_fig(def_grid)
        x_fig = comput_fig(x_seg)
        tar_fig = comput_fig(y_seg)

        writer.add_figure(
            'Grid',
            grid_fig,
            epoch
        )

        plt.close(grid_fig)

        writer.add_figure(
            'input',
            x_fig,
            epoch
        )

        plt.close(x_fig)

        writer.add_figure(
            'ground truth',
            tar_fig,
            epoch
        )

        plt.close(tar_fig)

        writer.add_figure(
            'prediction',
            pred_fig,
            epoch
        )

        plt.close(pred_fig)

        loss_all.reset()

    writer.close()


def comput_fig(img):
    img = (
        img.detach()
        .cpu()
        .numpy()[0, 0, 48:64, :, :]
    )

    fig = plt.figure(
        figsize=(12, 12),
        dpi=180
    )

    for i in range(img.shape[0]):
        plt.subplot(
            4,
            4,
            i + 1
        )

        plt.axis('off')

        plt.imshow(
            img[i, :, :],
            cmap='gray'
        )

    fig.subplots_adjust(
        wspace=0,
        hspace=0
    )

    return fig


def adjust_learning_rate(
    optimizer,
    epoch,
    MAX_EPOCHES,
    INIT_LR,
    power=0.9
):
    for param_group in optimizer.param_groups:
        param_group['lr'] = round(
            INIT_LR
            * np.power(
                1 - (epoch / MAX_EPOCHES),
                power
            ),
            8
        )


def mk_grid_img(
    grid_step,
    line_thickness=1,
    grid_sz=(160, 192, 224)
):
    grid_img = np.zeros(
        grid_sz
    )

    for j in range(
        0,
        grid_img.shape[1],
        grid_step
    ):
        grid_img[
            :,
            j + line_thickness - 1,
            :
        ] = 1

    for i in range(
        0,
        grid_img.shape[2],
        grid_step
    ):
        grid_img[
            :,
            :,
            i + line_thickness - 1
        ] = 1

    grid_img = grid_img[
        None,
        None,
        ...
    ]

    grid_img = torch.from_numpy(
        grid_img
    ).float().cuda()

    return grid_img


def save_checkpoint(
    state,
    save_dir='models/',
    filename='checkpoint.pth.tar',
    max_model_num=8
):
    os.makedirs(
        save_dir,
        exist_ok=True
    )

    torch.save(
        state,
        os.path.join(
            save_dir,
            filename
        )
    )

    model_lists = natsorted(
        glob.glob(
            os.path.join(
                save_dir,
                '*'
            )
        )
    )

    while len(model_lists) > max_model_num:
        os.remove(
            model_lists[0]
        )

        model_lists = natsorted(
            glob.glob(
                os.path.join(
                    save_dir,
                    '*'
                )
            )
        )


'''
GPU configuration
'''
GPU_iden = 0

GPU_num = torch.cuda.device_count()

print(
    'Number of GPU: '
    + str(GPU_num)
)

for GPU_idx in range(GPU_num):
    GPU_name = torch.cuda.get_device_name(
        GPU_idx
    )

    print(
        '     GPU #'
        + str(GPU_idx)
        + ': '
        + GPU_name
    )

if GPU_num == 0:
    raise RuntimeError(
        'CUDA GPU를 찾을 수 없습니다.'
    )

torch.cuda.set_device(
    GPU_iden
)

GPU_avai = torch.cuda.is_available()

print(
    'Currently using: '
    + torch.cuda.get_device_name(
        GPU_iden
    )
)

print(
    'If the GPU is available? '
    + str(GPU_avai)
)


main()

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

